In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold


# Ajustar estas listas a las columnas del dataset
numeric_features = ['age', 'fare']
categorical_features = ['pclass', 'sex', 'embarked']


# Pipeline para los atributos numéricos
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer())
])


# Pipeline para los atributos categóricos
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(fill_value='missing')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])


# Aplicamos un pipeline diferente a cada grupo de columnas
preprocessing = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])


# Pipeline completo: preprocesamiento + modelo
pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', DecisionTreeClassifier(random_state=42))
])


# Hiperparámetros del preprocesamiento y del modelo
param_grid = {
    # Preprocesamiento numérico
    'preprocessing__numeric__imputer__strategy': [
        'mean',
        'median'
    ],
    'preprocessing__numeric__imputer__add_indicator': [
        False,
        True
    ],

    # Preprocesamiento categórico
    'preprocessing__categorical__imputer__strategy': [
        'most_frequent',
        'constant'
    ],
    'preprocessing__categorical__onehot__drop': [
        None,
        'first'
    ],

    # Árbol de decisión
    'model__max_depth': [
        3,
        5,
        None
    ],
    'model__min_samples_leaf': [
        1,
        5
    ]
}


# Estrategia de cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    refit=True
)


# GridSearchCV realiza internamente la validación cruzada
grid_search.fit(X_train, y_train)


print("Mejores hiperparámetros:")
print(grid_search.best_params_)

print(f"\nMejor accuracy medio: {grid_search.best_score_:.3f}")


# Evaluación final sobre test
test_accuracy = grid_search.score(X_test, y_test)
print(f"Accuracy en test: {test_accuracy:.3f}")